# AgentAlign Lab — GPU Pipeline

**Run on Kaggle T4 x2 or Colab T4**

This notebook runs the complete GPU pipeline:
1. **Step 2**: Base model rollouts on train split → on-policy preference pairs
2. **Step 3**: DPO + SFT training with QLoRA
3. **Step 4**: Evaluation of base vs DPO vs SFT on held-out test split

Each step is checkpointed and resumable.

In [ ]:
# 0. Setup
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# 1. Extract handoff bundle and install
!unzip -o -q /kaggle/input/*/gpu_handoff.zip -d /kaggle/working/AgentAlign-Lab || echo 'Not a zip'
%cd /kaggle/working/AgentAlign-Lab
!pip install -e '.[ml,dashboard]' -q

In [ ]:
# 2. Verify tasks exist
import json, collections
for split in ['train', 'val', 'test']:
    tasks = [json.loads(l) for l in open(f'data/tasks/{split}.jsonl')]
    fam = collections.Counter(t['family'] for t in tasks)
    print(f'{split}: {len(tasks)} tasks')
    for k,v in sorted(fam.items()): print(f'  {k}: {v}')
    print()

## Step 2: On-Policy Rollouts

Run the BASE model (Qwen2.5-Coder-1.5B-Instruct) on the train split.
- 6 reps per task, temperature=0.8, max 8 steps, seed=42
- ~81 tasks × 6 reps = 486 trajectories
- Estimated time: 2-4 hours on T4

In [ ]:
# 2a. Run base model rollouts on TRAIN split
!python scripts/11_run_llm_rollouts.py \
    --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --split train \
    --reps 6 \
    --temperature 0.8 \
    --seed 42 \
    --max-steps 8 \
    --device cuda \
    --agent qwen_base \
    --resume

In [ ]:
# 2b. Analyze base model results
import json
from pathlib import Path
from collections import Counter, defaultdict

scored_dir = Path('data/trajectories/scored_llm_train')
trajectories = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if line.strip():
            trajectories.append(json.loads(line))

print(f'Total trajectories: {len(trajectories)}')

# Per-family pass rate
family_stats = defaultdict(lambda: {'total': 0, 'passed': 0})
for t in trajectories:
    task_id = t.get('task_id', '')
    family = '_'.join(task_id.rsplit('_', 1)[0].split('_')[:2]) if '_' in task_id else task_id
    vr = t.get('verifier_result', {})
    family_stats[family]['total'] += 1
    if vr.get('passed', False):
        family_stats[family]['passed'] += 1

print(f'\nBase pass rate per family:')
for fam in sorted(family_stats):
    s = family_stats[fam]
    rate = s['passed'] / max(s['total'], 1) * 100
    print(f'  {fam}: {s["passed"]}/{s["total"]} ({rate:.1f}%)')

# Preference pairs
pref_path = Path('data/preferences/dpo_llm_train.jsonl')
if pref_path.exists():
    pairs = [json.loads(l) for l in pref_path.read_text().splitlines() if l.strip()]
    margins = [p.get('score_margin', 0) for p in pairs]
    print(f'\nPreference pairs: {len(pairs)}')
    if margins:
        print(f'  Margin: min={min(margins):.1f}, max={max(margins):.1f}, mean={sum(margins)/len(margins):.1f}')
else:
    print('\nNo preference pairs yet.')

# Save analysis
analysis = {
    'total_trajectories': len(trajectories),
    'family_stats': {k: dict(v) for k, v in family_stats.items()},
    'n_pairs': len(pairs) if pref_path.exists() else 0,
    'margin_stats': {
        'min': min(margins) if margins else 0,
        'max': max(margins) if margins else 0,
        'mean': sum(margins)/len(margins) if margins else 0,
    } if pref_path.exists() and margins else {},
}
Path('outputs/evals').mkdir(parents=True, exist_ok=True)
Path('outputs/evals/base_rollout_analysis.json').write_text(json.dumps(analysis, indent=2))
print('\nAnalysis saved to outputs/evals/base_rollout_analysis.json')

## Step 3: Training

### 3a: DPO Training
QLoRA (4-bit NF4) with LoRA r=16 on Qwen2.5-Coder-1.5B-Instruct.

### 3b: SFT Baseline
Same data (chosen trajectories only), same compute budget.

In [ ]:
# 3a. DPO Training
import yaml, time, json
from pathlib import Path

# Update config to use LLM-generated pairs
config_path = Path('configs/dpo_qwen15_lora.yaml')
config = yaml.safe_load(config_path.read_text())

llm_train = Path('data/preferences/dpo_llm_train.jsonl')
if llm_train.exists():
    n_pairs = sum(1 for l in llm_train.read_text().splitlines() if l.strip())
    if n_pairs > 0:
        config['train_file'] = str(llm_train)
        print(f'Using {n_pairs} LLM-generated pairs for DPO training')
    else:
        print('LLM pairs empty, using scripted baseline pairs')
else:
    print('No LLM pairs found, using scripted baseline pairs')

config['output_dir'] = 'outputs/adapters/qwen_dpo_final'

# Write temp config
tmp_config = Path('configs/_dpo_run.yaml')
tmp_config.write_text(yaml.dump(config))

start_time = time.time()
from agentalign.train.dpo import run_dpo
run_dpo(config_path=str(tmp_config), dry_run=False)
dpo_wall_time = time.time() - start_time

import torch
dpo_meta = {
    'wall_time_seconds': round(dpo_wall_time, 1),
    'gpu_type': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unknown',
    'peak_memory_mb': round(torch.cuda.max_memory_allocated() / 1024**2, 1) if torch.cuda.is_available() else 0,
}
print(f'\nDPO training done in {dpo_wall_time:.0f}s')
print(f'Peak GPU memory: {dpo_meta["peak_memory_mb"]:.0f} MB')
Path('outputs/adapters/qwen_dpo_final/gpu_training_meta.json').write_text(json.dumps(dpo_meta, indent=2))

In [ ]:
# 3b. SFT Baseline Training
import yaml, time, json, torch
from pathlib import Path

config = yaml.safe_load(Path('configs/dpo_qwen15_lora.yaml').read_text())
llm_train = Path('data/preferences/dpo_llm_train.jsonl')
if llm_train.exists():
    config['train_file'] = str(llm_train)
config['output_dir'] = 'outputs/adapters/qwen_sft_final'

tmp_config = Path('configs/_sft_run.yaml')
tmp_config.write_text(yaml.dump(config))

torch.cuda.reset_peak_memory_stats()
start_time = time.time()
from agentalign.train.sft import run_sft
run_sft(config_path=str(tmp_config))
sft_wall_time = time.time() - start_time

sft_meta = {
    'wall_time_seconds': round(sft_wall_time, 1),
    'gpu_type': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unknown',
    'peak_memory_mb': round(torch.cuda.max_memory_allocated() / 1024**2, 1) if torch.cuda.is_available() else 0,
}
print(f'\nSFT training done in {sft_wall_time:.0f}s')
Path('outputs/adapters/qwen_sft_final').mkdir(parents=True, exist_ok=True)
Path('outputs/adapters/qwen_sft_final/gpu_training_meta.json').write_text(json.dumps(sft_meta, indent=2))

## Step 4: Held-Out Evaluation

Run base, SFT, and DPO on the test split (42 tasks × 4 reps each = 504 total trajectories).

In [ ]:
# 4a. Run all three models on test split
models = [
    ('qwen_base', None),
    ('qwen_sft', 'outputs/adapters/qwen_sft_final'),
    ('qwen_dpo', 'outputs/adapters/qwen_dpo_final'),
]

for agent_id, adapter in models:
    print(f'\n{"="*60}')
    print(f'Evaluating: {agent_id}')
    print(f'{"="*60}')
    
    cmd = (
        f'python scripts/02_run_agent.py '
        f'--split test --agent {agent_id} '
        f'--model Qwen/Qwen2.5-Coder-1.5B-Instruct '
        f'--out runs/llm_test '
        f'--repetitions 4 '
        f'--temperature 0.8 --seed 42 --max-steps 8 '
        f'--device cuda --resume'
    )
    if adapter:
        cmd += f' --adapter {adapter}'
    !{cmd}

In [ ]:
# 4b. Score all test trajectories
!python scripts/03_score_trajectories.py \
    --runs-dir runs/llm_test \
    --out-dir data/trajectories/scored_llm_test \
    --clear

In [ ]:
# 4c. Compute metrics and comparison
import json, numpy as np
from pathlib import Path
from collections import defaultdict

scored_dir = Path('data/trajectories/scored_llm_test')
trajectories = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if line.strip():
            trajectories.append(json.loads(line))

# Group by agent
by_agent = defaultdict(list)
for t in trajectories:
    by_agent[t.get('agent_id', 'unknown')].append(t)

# Compute metrics per agent
def compute_agent_metrics(trajs):
    total = len(trajs)
    passed = sum(1 for t in trajs if t.get('verifier_result', {}).get('passed', False))
    unsafe = sum(t.get('verifier_result', {}).get('unsafe_actions', 0) for t in trajs)
    failed_cmds = sum(t.get('verifier_result', {}).get('failed_commands', 0) for t in trajs)
    invalid = sum(t.get('verifier_result', {}).get('invalid_actions', 0) for t in trajs)
    avg_steps = np.mean([len(t.get('steps', [])) for t in trajs])
    return {
        'total': total,
        'passed': passed,
        'pass_rate': round(passed / max(total, 1), 4),
        'unsafe_actions': unsafe,
        'unsafe_rate': round(unsafe / max(total, 1), 4),
        'failed_commands': failed_cmds,
        'invalid_actions': invalid,
        'avg_steps': round(avg_steps, 2),
    }

results = {}
for agent, trajs in sorted(by_agent.items()):
    m = compute_agent_metrics(trajs)
    results[agent] = m
    print(f'{agent}: {m["passed"]}/{m["total"]} pass ({m["pass_rate"]*100:.1f}%), '
          f'unsafe={m["unsafe_actions"]}, invalid={m["invalid_actions"]}, '
          f'avg_steps={m["avg_steps"]}')

# Per-family breakdown
print('\n--- Per-family pass rates ---')
for agent in sorted(by_agent):
    trajs = by_agent[agent]
    family_stats = defaultdict(lambda: {'total': 0, 'passed': 0})
    for t in trajs:
        task_id = t.get('task_id', '')
        parts = task_id.rsplit('_', 1)
        family = parts[0] if len(parts) > 1 else task_id
        # Normalize: python_bugfix_001 -> python_bugfix
        family = '_'.join(family.split('_')[:2])
        family_stats[family]['total'] += 1
        if t.get('verifier_result', {}).get('passed', False):
            family_stats[family]['passed'] += 1
    print(f'\n{agent}:')
    for fam in sorted(family_stats):
        s = family_stats[fam]
        rate = s['passed'] / max(s['total'], 1) * 100
        print(f'  {fam}: {s["passed"]}/{s["total"]} ({rate:.1f}%)')
    results[agent]['per_family'] = {k: dict(v) for k, v in family_stats.items()}

# Bootstrap CI for pass-rate differences
def paired_bootstrap_ci(agent_a_trajs, agent_b_trajs, n_boot=10000, seed=42):
    """Paired per-task bootstrap 95% CI for pass-rate difference."""
    rng = np.random.RandomState(seed)
    
    # Group by task_id
    a_by_task = defaultdict(list)
    b_by_task = defaultdict(list)
    for t in agent_a_trajs:
        a_by_task[t['task_id']].append(t.get('verifier_result', {}).get('passed', False))
    for t in agent_b_trajs:
        b_by_task[t['task_id']].append(t.get('verifier_result', {}).get('passed', False))
    
    common_tasks = sorted(set(a_by_task) & set(b_by_task))
    if not common_tasks:
        return 0, 0, 0, 1.0
    
    # Per-task pass rates
    a_rates = np.array([np.mean(a_by_task[t]) for t in common_tasks])
    b_rates = np.array([np.mean(b_by_task[t]) for t in common_tasks])
    diffs = b_rates - a_rates  # positive = B better
    
    observed_diff = np.mean(diffs)
    
    # Bootstrap
    boot_diffs = []
    n = len(diffs)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        boot_diffs.append(np.mean(diffs[idx]))
    
    boot_diffs = np.array(boot_diffs)
    ci_lo = np.percentile(boot_diffs, 2.5)
    ci_hi = np.percentile(boot_diffs, 97.5)
    
    # p-value (two-sided)
    p_value = np.mean(np.abs(boot_diffs - np.mean(boot_diffs)) >= abs(observed_diff))
    
    return round(observed_diff, 4), round(ci_lo, 4), round(ci_hi, 4), round(p_value, 4)

# Compute CIs
if 'qwen_base' in by_agent and 'qwen_dpo' in by_agent:
    diff, ci_lo, ci_hi, p = paired_bootstrap_ci(by_agent['qwen_base'], by_agent['qwen_dpo'])
    print(f'\nDPO - Base: {diff:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}] p={p}')
    results['dpo_vs_base'] = {'diff': diff, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'p_value': p}

if 'qwen_base' in by_agent and 'qwen_sft' in by_agent:
    diff, ci_lo, ci_hi, p = paired_bootstrap_ci(by_agent['qwen_base'], by_agent['qwen_sft'])
    print(f'SFT - Base: {diff:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}] p={p}')
    results['sft_vs_base'] = {'diff': diff, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'p_value': p}

if 'qwen_sft' in by_agent and 'qwen_dpo' in by_agent:
    diff, ci_lo, ci_hi, p = paired_bootstrap_ci(by_agent['qwen_sft'], by_agent['qwen_dpo'])
    print(f'DPO - SFT: {diff:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}] p={p}')
    results['dpo_vs_sft'] = {'diff': diff, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'p_value': p}

# Save results
Path('outputs/evals').mkdir(parents=True, exist_ok=True)
Path('outputs/evals/eval_results.json').write_text(json.dumps(results, indent=2))
print('\nResults saved to outputs/evals/eval_results.json')

In [ ]:
# 4d. Failure taxonomy — 10 DPO test failures
import json
from pathlib import Path

scored_dir = Path('data/trajectories/scored_llm_test')
failures = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if not line.strip():
            continue
        t = json.loads(line)
        if t.get('agent_id') == 'qwen_dpo' and not t.get('verifier_result', {}).get('passed', False):
            failures.append(t)

print(f'Total DPO failures on test: {len(failures)}')

# Sample 10
sample = failures[:10]
taxonomy = []
for i, f in enumerate(sample):
    vr = f.get('verifier_result', {})
    tags = vr.get('failure_tags', [])
    label = vr.get('failure_label', 'unknown')
    n_steps = len(f.get('steps', []))
    last_action = f['steps'][-1].get('action', 'none') if f.get('steps') else 'none'
    taxonomy.append({
        'task_id': f.get('task_id'),
        'failure_label': label,
        'failure_tags': tags,
        'n_steps': n_steps,
        'last_action': last_action,
        'score': vr.get('score', 0),
    })
    print(f'{i+1}. {f.get("task_id")}: {label} ({tags}), {n_steps} steps, last={last_action}')

Path('outputs/evals/failure_taxonomy.json').write_text(json.dumps(taxonomy, indent=2))
print('\nTaxonomy saved to outputs/evals/failure_taxonomy.json')

In [ ]:
# 5. Package all outputs for download
import shutil

# Create export directory
export_dir = Path('/kaggle/working/export')
export_dir.mkdir(exist_ok=True)

# Copy key artifacts
for src in [
    'outputs/adapters/qwen_dpo_final',
    'outputs/adapters/qwen_sft_final',
    'outputs/evals',
    'data/preferences/dpo_llm_train.jsonl',
    'data/trajectories/scored_llm_train',
    'data/trajectories/scored_llm_test',
    'runs/llm_train',
    'runs/llm_test',
]:
    src_path = Path(src)
    if not src_path.exists():
        print(f'  SKIP (not found): {src}')
        continue
    dst = export_dir / src
    if src_path.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_path, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dst)
    print(f'  {src}')

# Create zip
shutil.make_archive('/kaggle/working/agentalign_results', 'zip', export_dir)
print(f'\nDownload /kaggle/working/agentalign_results.zip')
print(f'   Then run: python scripts/10_import_colab_artifacts.py --zip-path agentalign_results.zip')